# Etapa 3 — Regressão OLS

Estimação do modelo de mínimos quadrados ordinários (MQO) para analisar
os determinantes do crescimento do PIB municipal em Mato Grosso do Sul
(2019–2023). Replicamos o modelo de Faganello & Souza (2024) adaptado
ao estado de MS.

**Variável dependente (Y):** `crescimento_pib`  
**Variáveis explicativas (X1–X9):** log PIB per capita, investimento/PIB,
gasto pessoal/PIB, receita corrente/PIB, crescimento populacional,
área/habitante, distância da capital, dummy pequeno município,
interação dummy × pessoal.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.diagnostic import het_white, linear_reset
import scipy.stats as stats
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Carga e preparação dos dados

In [2]:
def localizar_outputs() -> Path:
    cwd = Path.cwd().resolve()
    candidatos = [cwd, cwd.parent, cwd / 'Etapa3' / 'Material_entrega', cwd.parent.parent]
    for c in candidatos:
        p = c / 'outputs' / 'base_etapa3_ms.csv'
        if p.exists():
            return c / 'outputs'
    raise FileNotFoundError('Nao encontrei outputs/base_etapa3_ms.csv')

OUTPUTS = localizar_outputs()
base = pd.read_csv(OUTPUTS / 'base_etapa3_ms.csv', sep=';', decimal=',')

VARIAVEL_Y = 'crescimento_pib'
VARIAVEIS_X = [
    'log_pib_per_capita',
    'investimento_pib',
    'gasto_pessoal_pib',
    'receita_corrente_pib',
    'crescimento_populacao',
    'area_por_habitante',
    'distancia_capital_km',
    'dummy_pequeno_municipio',
    'interacao_pequeno_pessoal',
]
ROTULOS_X = {
    'log_pib_per_capita':        'X1 — log PIB per capita',
    'investimento_pib':          'X2 — Investimento / PIB',
    'gasto_pessoal_pib':         'X3 — Gasto Pessoal / PIB',
    'receita_corrente_pib':      'X4 — Receita Corrente / PIB',
    'crescimento_populacao':     'X5 — Crescimento Populacional',
    'area_por_habitante':        'X6 — Area por Habitante',
    'distancia_capital_km':      'X7 — Distancia da Capital (km)',
    'dummy_pequeno_municipio':   'X8 — Dummy Pequeno Municipio',
    'interacao_pequeno_pessoal': 'X9 — Interacao (Dummy x Pessoal)',
}

# Amostra de regressao: remover NaN nas variaveis do modelo
df_reg = base[[VARIAVEL_Y] + VARIAVEIS_X].dropna().copy()
print(f'Observacoes totais na base : {len(base)}')
print(f'Observacoes na regressao   : {len(df_reg)}')
print(f'Removidas por NaN          : {len(base) - len(df_reg)}')
print(f'Municipios cobertos        : {base.loc[df_reg.index, "cod_ibge"].nunique() if "cod_ibge" in base.columns else "n/d"}')
df_reg.describe().T

Observacoes totais na base : 395
Observacoes na regressao   : 395
Removidas por NaN          : 0
Municipios cobertos        : 79


,count,mean,std,min,25%,50%,75%,max
crescimento_pib,395.0000,0.1545,0.1951,-0.5817,0.0308,0.1418,0.2667,1.0947
log_pib_per_capita,395.0000,10.7900,0.5715,9.4574,10.4023,10.7376,11.1485,12.7793
investimento_pib,395.0000,0.0139,0.0114,0.0009,0.0060,0.0109,0.0172,0.0726
gasto_pessoal_pib,395.0000,0.0659,0.0291,0.0144,0.0430,0.0615,0.0836,0.1664
receita_corrente_pib,395.0000,0.1422,0.0658,0.0326,0.0924,0.1289,0.1735,0.3604
crescimento_populacao,395.0000,0.0053,0.0100,-0.0164,-0.0014,0.0036,0.0109,0.0438
area_por_habitante,395.0000,0.2655,0.2754,0.0085,0.0868,0.1622,0.3873,1.4019
distancia_capital_km,395.0000,277.6889,117.2589,0.0000,198.0000,289.2400,368.5400,469.4100
dummy_pequeno_municipio,395.0000,0.2532,0.4354,0.0000,0.0000,0.0000,1.0000,1.0000
interacao_pequeno_pessoal,395.0000,0.0176,0.0341,0.0000,0.0000,0.0000,0.0145,0.1362


## 2. Amostra de regressão

A regressão usa dados empilhados (`pooled OLS`) para os 5 exercícios fiscais
(2019–2023). Cada linha é um município-ano. Não aplicamos efeitos fixos
nesta etapa; um modelo com efeitos fixos de município (within estimator)
seria o próximo passo natural.

In [3]:
# Distribuicao da amostra por ano
obs_ano = base.dropna(subset=[VARIAVEL_Y] + VARIAVEIS_X).groupby('ano').size().reset_index(name='obs')
print('Observacoes por ano:')
print(obs_ano.to_string(index=False))

Observacoes por ano:
 ano  obs
2019   79
2020   79
2021   79
2022   79
2023   79


## 3. Regressão OLS — modelo completo (dados empilhados 2019–2023)

Estimamos por MQO o modelo:

$$Y_i = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \beta_4 X_4 + \beta_5 X_5 + \beta_6 X_6 + \beta_7 X_7 + \beta_8 X_8 + \beta_9 X_9 + \varepsilon_i$$

Os erros-padrão são clássicos (não robustos). Use `cov_type='HC3'` no `.fit()` para erros robustos à heterocedasticidade após o teste de White.

In [4]:
formula = (
    'crescimento_pib ~ '
    'log_pib_per_capita + investimento_pib + gasto_pessoal_pib + '
    'receita_corrente_pib + crescimento_populacao + area_por_habitante + '
    'distancia_capital_km + dummy_pequeno_municipio + interacao_pequeno_pessoal'
)

modelo = smf.ols(formula=formula, data=df_reg).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:        crescimento_pib   R-squared:                       0.103
Model:                            OLS   Adj. R-squared:                  0.082
Method:                 Least Squares   F-statistic:                     4.915
Date:                Tue, 26 May 2026   Prob (F-statistic):           2.96e-06
Time:                        19:28:23   Log-Likelihood:                 106.97
No. Observations:                 395   AIC:                            -193.9
Df Residuals:                     385   BIC:                            -154.2
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [5]:
formula = (
    'crescimento_pib ~ '
    'log_pib_per_capita + investimento_pib + gasto_pessoal_pib + '
    'receita_corrente_pib + crescimento_populacao + area_por_habitante + '
    'distancia_capital_km + dummy_pequeno_municipio + interacao_pequeno_pessoal'
)

modelo = smf.ols(formula=formula, data=df_reg).fit(cov_type='HC3') #hc3
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:        crescimento_pib   R-squared:                       0.103
Model:                            OLS   Adj. R-squared:                  0.082
Method:                 Least Squares   F-statistic:                     3.615
Date:                Tue, 26 May 2026   Prob (F-statistic):           0.000242
Time:                        19:28:23   Log-Likelihood:                 106.97
No. Observations:                 395   AIC:                            -193.9
Df Residuals:                     385   BIC:                            -154.2
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [6]:
# Tabela de resultados formatada
coef_df = pd.DataFrame({
    'Variavel': ['Intercepto'] + [ROTULOS_X.get(v, v) for v in VARIAVEIS_X],
    'Coeficiente': modelo.params.values,
    'Erro Padrao': modelo.bse.values,
    'Estatistica t': modelo.tvalues.values,
    'p-valor': modelo.pvalues.values,
    'IC 2.5%': modelo.conf_int()[0].values,
    'IC 97.5%': modelo.conf_int()[1].values,
})
coef_df['Signif.'] = coef_df['p-valor'].apply(
    lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else ''))
)

print(f'R2          : {modelo.rsquared:.4f}')
print(f'R2 ajustado : {modelo.rsquared_adj:.4f}')
print(f'F-estatistica: {modelo.fvalue:.4f}  (p={modelo.f_pvalue:.4f})')
print(f'AIC         : {modelo.aic:.2f}')
print(f'BIC         : {modelo.bic:.2f}')
print(f'Observacoes : {int(modelo.nobs)}')
print()
coef_df

R2          : 0.1031
R2 ajustado : 0.0821
F-estatistica: 3.6150  (p=0.0002)
AIC         : -193.94
BIC         : -154.15
Observacoes : 395



,Variavel,Coeficiente,Erro Padrao,Estatistica t,p-valor,IC 2.5%,IC 97.5%,Signif.
0,Intercepto,-0.8770,0.4437,-1.9765,0.0481,-1.7468,-0.0073,**
1,X1 — log PIB per capita,0.1012,0.0384,2.6342,0.0084,0.0259,0.1764,***
2,X2 — Investimento / PIB,1.7106,1.0106,1.6927,0.0905,-0.2701,3.6913,*
3,X3 — Gasto Pessoal / PIB,-1.3570,1.1210,-1.2105,0.2261,-3.5541,0.8401,
4,X4 — Receita Corrente / PIB,0.5244,0.5170,1.0143,0.3104,-0.4889,1.5376,
5,X5 — Crescimento Populacional,-4.1055,1.1653,-3.5231,0.0004,-6.3896,-1.8215,***
6,X6 — Area por Habitante,-0.0404,0.0443,-0.9120,0.3618,-0.1271,0.0464,
7,X7 — Distancia da Capital (km),-0.0001,0.0001,-0.9994,0.3176,-0.0003,0.0001,
8,X8 — Dummy Pequeno Municipio,0.0162,0.0786,0.2058,0.8370,-0.1378,0.1701,
9,X9 — Interacao (Dummy x Pessoal),-0.8940,0.9342,-0.9569,0.3386,-2.7250,0.9371,


## 4. Diagnóstico de multicolinearidade — VIF

O Fator de Inflação da Variância (VIF) mede o quanto a variância de cada
coeficiente aumenta por conta da correlação com os demais regressores.
Regra prática: VIF < 5 indica multicolinearidade aceitável; VIF > 10 é crítico.

In [7]:
X_vif = sm.add_constant(df_reg[VARIAVEIS_X])
vif_df = pd.DataFrame({
    'Variavel': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
vif_df['Interpretacao'] = vif_df['VIF'].apply(
    lambda v: 'OK (< 5)' if v < 5 else ('Moderado (5-10)' if v < 10 else 'ALTO (> 10)')
)
vif_df.sort_values('VIF', ascending=False)

,Variavel,VIF,Interpretacao
0,const,1496.9925,ALTO (> 10)
4,receita_corrente_pib,15.5717,ALTO (> 10)
3,gasto_pessoal_pib,14.3872,ALTO (> 10)
9,interacao_pequeno_pessoal,7.9559,Moderado (5-10)
8,dummy_pequeno_municipio,6.9751,Moderado (5-10)
1,log_pib_per_capita,3.6788,OK (< 5)
2,investimento_pib,1.8207,OK (< 5)
6,area_por_habitante,1.7332,OK (< 5)
5,crescimento_populacao,1.3990,OK (< 5)
7,distancia_capital_km,1.1594,OK (< 5)


In [8]:
# RODAR SEM RECEITA- receita_corrente_pib	15.9710	

## 5. Normalidade dos resíduos — Jarque-Bera

H0: Os resíduos seguem distribuição normal (assimetria = 0, curtose = 3).  
Rejeitar H0 indica que os resíduos não são normais, o que afeta a validade
dos testes t e F em amostras pequenas (n > 200: o Teorema Central do Limite mitiga o problema).

In [9]:
jb_stat, jb_pval, jb_skew, jb_kurt = jarque_bera(modelo.resid)
print(f'Jarque-Bera: estatistica={jb_stat:.4f}  p-valor={jb_pval:.4f}')
print(f'Assimetria  : {jb_skew:.4f}')
print(f'Curtose     : {jb_kurt:.4f}')
print(f'Conclusao   : {"H0 nao rejeitada (residuos normais)" if jb_pval > 0.05 else "H0 REJEITADA (residuos nao normais)"} ao nivel 5%')

# Q-Q plot
(osm, osr), (slope, intercept, r) = stats.probplot(modelo.resid)
fig_qq = go.Figure()
fig_qq.add_scatter(x=osm, y=osr, mode='markers', name='Residuos',
                   marker=dict(color='#2E75B6', size=5, opacity=0.7))
fig_qq.add_scatter(x=[osm[0], osm[-1]],
                   y=[slope*osm[0]+intercept, slope*osm[-1]+intercept],
                   mode='lines', name='Linha normal',
                   line=dict(color='#E74C3C', width=2))
fig_qq.update_layout(title='Q-Q Plot dos Residuos', plot_bgcolor='white',
                     xaxis_title='Quantis teoricos', yaxis_title='Quantis amostrais',
                     legend=dict(orientation='h', y=-0.2))
fig_qq.show()

Jarque-Bera: estatistica=70.3213  p-valor=0.0000
Assimetria  : 0.4105
Curtose     : 4.8970
Conclusao   : H0 REJEITADA (residuos nao normais) ao nivel 5%


## 6. Heterocedasticidade — Teste de White

H0: Os erros são homocedasticos (variância constante).  
Rejeitar H0 indica heterocedasticidade: os coeficientes ainda são
não viesados, mas os erros-padrão clássicos são incorretos. Solução:
usar erros robustos (HC3) na estimação.

In [10]:
white_stat, white_pval, white_f, white_f_pval = het_white(modelo.resid, modelo.model.exog)
print(f'Teste de White:')
print(f'  Estatistica LM  : {white_stat:.4f}')
print(f'  p-valor (LM)    : {white_pval:.4f}')
print(f'  Estatistica F   : {white_f:.4f}')
print(f'  p-valor (F)     : {white_f_pval:.4f}')
print(f'  Conclusao       : {"H0 nao rejeitada (homocedasticidade)" if white_pval > 0.05 else "H0 REJEITADA (heterocedasticidade detectada)"} ao nivel 5%')

# Se heterocedasticidade presente, reestima com erros robustos
if white_pval <= 0.05:
    modelo_hc3 = smf.ols(formula=formula, data=df_reg).fit(cov_type='HC3')
    print()
    print('--- Reestimacao com erros robustos HC3 ---')
    print(modelo_hc3.summary())

Teste de White:
  Estatistica LM  : 105.3071
  p-valor (LM)    : 0.0000
  Estatistica F   : 2.5010
  p-valor (F)     : 0.0000
  Conclusao       : H0 REJEITADA (heterocedasticidade detectada) ao nivel 5%

--- Reestimacao com erros robustos HC3 ---
                            OLS Regression Results                            
Dep. Variable:        crescimento_pib   R-squared:                       0.103
Model:                            OLS   Adj. R-squared:                  0.082
Method:                 Least Squares   F-statistic:                     3.615
Date:                Tue, 26 May 2026   Prob (F-statistic):           0.000242
Time:                        19:28:28   Log-Likelihood:                 106.97
No. Observations:                 395   AIC:                            -193.9
Df Residuals:                     385   BIC:                            -154.2
Df Model:                           9                                         
Covariance Type:                  HC3     

## 7. Autocorrelação dos resíduos — Durbin-Watson

H0: Não há autocorrelação serial dos resíduos (DW próximo de 2).  
DW < 1,5 indica autocorrelação positiva; DW > 2,5 indica negativa.
Em dados de corte transversal anuais (pooled), este teste é menos crítico
do que em séries temporais.

In [11]:
dw_stat = durbin_watson(modelo.resid)
print(f'Durbin-Watson: {dw_stat:.4f}')
if dw_stat < 1.5:
    print('  -> Possivel autocorrelacao positiva')
elif dw_stat > 2.5:
    print('  -> Possivel autocorrelacao negativa')
else:
    print('  -> Sem evidencia de autocorrelacao serial')

Durbin-Watson: 1.8454
  -> Sem evidencia de autocorrelacao serial


## 8. Especificação do modelo — Ramsey RESET

H0: O modelo está corretamente especificado (sem variáveis omitidas ou
forma funcional incorreta). Rejeitar H0 sugere que o modelo pode precisar
de termos quadráticos, cúbicos ou outras variáveis.

In [12]:
reset_result = linear_reset(modelo, power=3, use_f=True)
print(f'Ramsey RESET (potencias 2 e 3):')
print(f'  Estatistica F: {reset_result.statistic:.4f}')
print(f'  p-valor      : {reset_result.pvalue:.4f}')
print(f'  Conclusao    : {"H0 nao rejeitada (especificacao adequada)" if reset_result.pvalue > 0.05 else "H0 REJEITADA (possivel misspecification)"} ao nivel 5%')

Ramsey RESET (potencias 2 e 3):
  Estatistica F: 2.3810
  p-valor      : 0.0938
  Conclusao    : H0 nao rejeitada (especificacao adequada) ao nivel 5%


## 9. Teste de Wald — hipótese dos pequenos municípios

Testamos a hipótese conjunta de que X8 (`dummy_pequeno_municipio`) e
X9 (`interacao_pequeno_pessoal`) são **simultaneamente iguais a zero**,
ou seja, de que o porte do município não afeta o crescimento do PIB.

H0: β₈ = 0 e β₉ = 0

In [13]:
wald = modelo.wald_test("dummy_pequeno_municipio = 0, interacao_pequeno_pessoal = 0")
wald_stat = float(np.squeeze(wald.statistic))
wald_pval = float(np.squeeze(wald.pvalue))
print(f"Teste de Wald (X8 = X9 = 0):")
print(f"  Estatistica F: {wald_stat:.4f}")
print(f"  p-valor      : {wald_pval:.4f}")
conclusao_wald = "H0 nao rejeitada (X8 e X9 nao significativos conjuntamente)" if wald_pval > 0.05 else "H0 REJEITADA (efeito de porte identificado conjuntamente)"
print(f"  Conclusao    : {conclusao_wald} ao nivel 5%")

Teste de Wald (X8 = X9 = 0):
  Estatistica F: 3.7823
  p-valor      : 0.1509
  Conclusao    : H0 nao rejeitada (X8 e X9 nao significativos conjuntamente) ao nivel 5%


C:\Users\marco\miniconda3\envs\work\Lib\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(


## 10. Tabela resumo dos testes diagnósticos

In [14]:
resumo_diag = pd.DataFrame([
    {"Teste": "Jarque-Bera (normalidade)",   "Estatistica": round(jb_stat, 4),                        "p-valor": round(jb_pval, 4),    "H0": "Residuos normais",             "Resultado": "Nao rejeitada" if jb_pval > 0.05 else "REJEITADA"},
    {"Teste": "White (heterocedasticidade)", "Estatistica": round(white_stat, 4),                      "p-valor": round(white_pval, 4), "H0": "Homocedasticidade",            "Resultado": "Nao rejeitada" if white_pval > 0.05 else "REJEITADA"},
    {"Teste": "Durbin-Watson (autocorr.)",   "Estatistica": round(dw_stat, 4),                         "p-valor": None,                 "H0": "Sem autocorrelacao (DW~2)",    "Resultado": "OK" if 1.5 <= dw_stat <= 2.5 else "Atencao"},
    {"Teste": "Ramsey RESET (especif.)",     "Estatistica": round(float(reset_result.statistic), 4),   "p-valor": round(reset_result.pvalue, 4), "H0": "Modelo bem especificado", "Resultado": "Nao rejeitada" if reset_result.pvalue > 0.05 else "REJEITADA"},
    {"Teste": "Wald (X8=X9=0)",             "Estatistica": round(wald_stat, 4),                       "p-valor": round(wald_pval, 4),  "H0": "Porte nao afeta crescimento", "Resultado": "Nao rejeitada" if wald_pval > 0.05 else "REJEITADA"},
])
resumo_diag

,Teste,Estatistica,p-valor,H0,Resultado
0,Jarque-Bera (normalidade),70.3213,0.0000,Residuos normais,REJEITADA
1,White (heterocedasticidade),105.3071,0.0000,Homocedasticidade,REJEITADA
2,Durbin-Watson (autocorr.),1.8454,NaN,Sem autocorrelacao (DW~2),OK
3,Ramsey RESET (especif.),2.3810,0.0938,Modelo bem especificado,Nao rejeitada
4,Wald (X8=X9=0),3.7823,0.1509,Porte nao afeta crescimento,Nao rejeitada


## 11. Diagnóstico visual dos resíduos

In [15]:
fitted = modelo.fittedvalues
resid  = modelo.resid

# Residuos vs Valores ajustados
fig_rv = px.scatter(
    x=fitted, y=resid,
    labels={'x': 'Valores ajustados', 'y': 'Residuos'},
    title='Residuos vs Valores Ajustados',
    opacity=0.65,
)
fig_rv.add_hline(y=0, line_dash='dash', line_color='red', line_width=1.5)
fig_rv.update_layout(plot_bgcolor='white', yaxis=dict(gridcolor='#eeeeee'))
fig_rv.show()

# Histograma dos residuos
fig_rh = px.histogram(x=resid, nbins=25, title='Distribuicao dos Residuos',
                      labels={'x': 'Residuo', 'y': 'Frequencia'})
fig_rh.update_layout(plot_bgcolor='white', yaxis=dict(gridcolor='#eeeeee'))
fig_rh.show()

## 12. Regressões por ano (2019–2023)

Estimamos o mesmo modelo para cada exercício fiscal separadamente
(79 observações por ano, 9 regressores). Os resultados servem para
avaliar a estabilidade dos coeficientes ao longo do tempo.

In [16]:
resultados_ano = []
for ano in sorted(df_reg.index.map(lambda i: base.loc[i, 'ano'] if i in base.index else None).dropna().unique() if 'ano' in base.columns else []):
    pass  # substituido abaixo

# Merge do ano de volta
df_reg2 = df_reg.copy()
df_reg2['ano'] = base.loc[df_reg2.index, 'ano'].values

resultados_ano = []
for ano in sorted(df_reg2['ano'].unique()):
    df_ano = df_reg2[df_reg2['ano'] == ano]
    if len(df_ano) < len(VARIAVEIS_X) + 2:
        continue
    m = smf.ols(formula=formula, data=df_ano).fit()
    row = {'Ano': int(ano), 'N': int(m.nobs), 'R2': round(m.rsquared, 4),
           'R2_adj': round(m.rsquared_adj, 4), 'F': round(m.fvalue, 4),
           'p_F': round(m.f_pvalue, 4), 'AIC': round(m.aic, 2)}
    for v in VARIAVEIS_X:
        row[v] = round(m.params.get(v, float('nan')), 4)
        row[f't_{v}'] = round(m.tvalues.get(v, float('nan')), 4)
    resultados_ano.append(row)

res_ano_df = pd.DataFrame(resultados_ano)
res_ano_df

,Ano,N,R2,R2_adj,F,p_F,AIC,log_pib_per_capita,t_log_pib_per_capita,investimento_pib,t_investimento_pib,gasto_pessoal_pib,t_gasto_pessoal_pib,receita_corrente_pib,t_receita_corrente_pib,crescimento_populacao,t_crescimento_populacao,area_por_habitante,t_area_por_habitante,distancia_capital_km,t_distancia_capital_km,dummy_pequeno_municipio,t_dummy_pequeno_municipio,interacao_pequeno_pessoal,t_interacao_pequeno_pessoal
0,2019,79,0.1420,0.0301,1.2687,0.2697,-105.4200,-0.0225,-0.3967,-2.5704,-1.5569,-3.3059,-1.5665,1.2992,1.2841,-1.6680,-0.9551,0.0973,1.4848,0.0001,0.7127,-0.1436,-1.6701,0.8507,0.8295
1,2020,79,0.2700,0.1748,2.8355,0.0067,-65.6000,0.1145,1.3616,-0.8111,-0.4163,-4.0421,-1.5740,1.3690,1.3201,-6.6717,-3.0475,-0.1770,-2.0820,-0.0000,-0.1165,-0.0481,-0.4365,1.5259,1.0669
2,2021,79,0.1194,0.0045,1.0395,0.4183,-65.5400,0.0017,0.0211,0.6822,0.2291,3.5924,1.4981,-2.3442,-1.8851,-2.2890,-1.0609,0.0719,0.7834,0.0002,0.9936,0.0192,0.1818,0.2253,0.1361
3,2022,79,0.2946,0.2026,3.2016,0.0027,-20.1100,0.0632,0.6117,3.2468,1.0528,-3.2021,-0.9404,1.4155,0.8837,2.7358,0.8829,0.0331,0.2547,-0.0006,-3.1003,0.2537,1.7651,-5.2274,-2.4808
4,2023,79,0.2700,0.1748,2.8357,0.0067,-27.4600,0.2056,1.6778,6.6308,2.4005,-2.3184,-0.7853,1.4530,0.9980,-10.6843,-3.6429,-0.2106,-1.4748,0.0002,1.1933,-0.0203,-0.1537,-0.8265,-0.4525


In [17]:
# ADICIONAR test t

In [18]:
# Grafico: evolucao dos coeficientes por ano
coef_long = res_ano_df[['Ano'] + VARIAVEIS_X].melt(id_vars='Ano', var_name='variavel', value_name='coef')
coef_long['variavel'] = coef_long['variavel'].map(ROTULOS_X)

fig_coef = px.line(
    coef_long, x='Ano', y='coef', color='variavel', markers=True,
    title='Evolucao dos coeficientes OLS por ano (2019-2023)',
    labels={'coef': 'Coeficiente', 'variavel': 'Variavel'},
)
fig_coef.add_hline(y=0, line_dash='dot', line_color='gray', line_width=1)
fig_coef.update_xaxes(tickmode='linear', dtick=1)
fig_coef.update_layout(plot_bgcolor='white', yaxis=dict(gridcolor='#eeeeee'),
                       legend=dict(orientation='h', y=-0.35, x=0))
fig_coef.show()

## 13. Seleção de modelo

Comparamos três especificações pelo AIC/BIC para identificar o modelo
mais parcimonioso:

- **M1 — Completo**: todos os 9 regressores  
- **M2 — Sem interação**: X1–X8 (remove X9)  
- **M3 — Sem porte**: X1–X7 (remove X8 e X9)  
- **M4 - Sem receitas**: remove X4 (Receita Corrente / PIB)  

In [19]:
# M4 incluido na comparacao abaixo: modelo sem receita_corrente_pib

In [20]:
especificacoes = {
    'M1 — Completo (X1-X9)': formula,
    'M2 — Sem interacao (X1-X8)': formula.replace(' + interacao_pequeno_pessoal', ''),
    'M3 — Sem porte (X1-X7)': formula.replace(' + dummy_pequeno_municipio + interacao_pequeno_pessoal', ''),
    'M4 - Sem receitas (sem X4)': formula.replace('receita_corrente_pib + ', ''),
}

sel_rows = []
for nome, f in especificacoes.items():
    m = smf.ols(formula=f, data=df_reg).fit()
    row = {
        'Modelo': nome, 'k': int(m.df_model + 1),
        'R2_adj': round(m.rsquared_adj, 4),
        'AIC': round(m.aic, 2), 'BIC': round(m.bic, 2),
        'F': round(m.fvalue, 4), 'p_F': round(m.f_pvalue, 4),
    }
    for v in VARIAVEIS_X:
        row[f't_{v}'] = round(m.tvalues.get(v, float('nan')), 4)
    sel_rows.append(row)

sel_df = pd.DataFrame(sel_rows)
print('Menor AIC e melhor:')
sel_df

Menor AIC e melhor:


,Modelo,k,R2_adj,AIC,BIC,F,p_F,t_log_pib_per_capita,t_investimento_pib,t_gasto_pessoal_pib,t_receita_corrente_pib,t_crescimento_populacao,t_area_por_habitante,t_distancia_capital_km,t_dummy_pequeno_municipio,t_interacao_pequeno_pessoal
0,M1 — Completo (X1-X9),10,0.0821,-193.9400,-154.1500,4.9153,0.0000,3.2001,1.5386,-1.1045,0.9281,-3.6854,-0.8966,-1.0301,0.2830,-1.1469
1,M2 — Sem interacao (X1-X8),9,0.0813,-194.5900,-158.7800,5.3609,0.0000,3.3375,1.3216,-1.0339,0.7045,-3.8020,-0.8172,-0.8265,-1.4755,NaN
2,M3 — Sem porte (X1-X7),8,0.0785,-194.3700,-162.5400,5.7981,0.0000,3.0246,1.0881,-0.8128,0.3275,-3.6151,-0.9663,-0.5007,NaN,NaN
3,M4 - Sem receitas (sem X4),9,0.0824,-195.0600,-159.2500,5.4240,0.0000,3.1653,1.8897,-0.5993,NaN,-3.6888,-0.6762,-0.9531,0.2309,-0.9753


## 14. Exportação dos resultados para Excel

In [21]:
borda = {'left': 'thin', 'right': 'thin', 'top': 'thin', 'bottom': 'thin'}

def estilo_cab(ws, row):
    for cell in ws[row]:
        cell.font   = Font(bold=True, color='FFFFFF', size=10)
        cell.fill   = PatternFill('solid', fgColor='1F3864')
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = Border(**{s: Side(style='thin', color='BFBFBF') for s in borda})

def auto_width(ws, min_w=10, max_w=40):
    for col in ws.columns:
        w = max(len(str(c.value or '')) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(w+2, min_w), max_w)

wb = openpyxl.Workbook()

# --- Aba Coeficientes ---
ws1 = wb.active
ws1.title = 'Coeficientes'
ws1.append(list(coef_df.columns))
estilo_cab(ws1, 1)
for _, row in coef_df.iterrows():
    ws1.append(list(row))
auto_width(ws1)

# --- Aba VIF ---
ws2 = wb.create_sheet('VIF')
ws2.append(list(vif_df.columns))
estilo_cab(ws2, 1)
for _, row in vif_df.iterrows():
    ws2.append(list(row))
auto_width(ws2)

# --- Aba Diagnosticos ---
ws3 = wb.create_sheet('Diagnosticos')
ws3.append(list(resumo_diag.columns))
estilo_cab(ws3, 1)
for _, row in resumo_diag.iterrows():
    ws3.append(list(row))
auto_width(ws3)

# --- Aba Por Ano ---
ws4 = wb.create_sheet('Por Ano')
ws4.append(list(res_ano_df.columns))
estilo_cab(ws4, 1)
for _, row in res_ano_df.iterrows():
    ws4.append(list(row))
auto_width(ws4)

# --- Aba Selecao ---
ws5 = wb.create_sheet('Selecao Modelo')
ws5.append(list(sel_df.columns))
estilo_cab(ws5, 1)
for _, row in sel_df.iterrows():
    ws5.append(list(row))
auto_width(ws5)

xl_path = OUTPUTS / 'regressao_ols_resultados.xlsx'
wb.save(xl_path)
print(f'Excel salvo: {xl_path}')
print(f'  Abas: Coeficientes, VIF, Diagnosticos, Por Ano, Selecao Modelo')

Excel salvo: C:\Users\marco\Documents\PUC-MINAS\4º Semestre\Eixo 4 - Projeto Mini Ministério da Fazenda\Projeto-Mini-Min-Fazenda\Etapa3\Material_entrega\outputs\regressao_ols_resultados.xlsx
  Abas: Coeficientes, VIF, Diagnosticos, Por Ano, Selecao Modelo


In [22]:
print('=== RESUMO DA REGRESSAO ===')
print(f'Equacao: Y ~ X1 + X2 + X3 + X4 + X5 + X6 + X7 + X8 + X9')
print(f'N = {int(modelo.nobs)}  |  R2 = {modelo.rsquared:.4f}  |  R2adj = {modelo.rsquared_adj:.4f}')
print(f'F = {modelo.fvalue:.4f}  |  p(F) = {modelo.f_pvalue:.4f}')
print(f'AIC = {modelo.aic:.2f}  |  BIC = {modelo.bic:.2f}')
print()
print('Coeficientes significativos (p < 0.10):')
sig = coef_df[coef_df['Signif.'] != '']
for _, r in sig.iterrows():
    print(f'  {r["Variavel"]:<40} beta={r["Coeficiente"]:+.4f}  p={r["p-valor"]:.4f}  {r["Signif."]}')

=== RESUMO DA REGRESSAO ===
Equacao: Y ~ X1 + X2 + X3 + X4 + X5 + X6 + X7 + X8 + X9
N = 395  |  R2 = 0.1031  |  R2adj = 0.0821
F = 3.6150  |  p(F) = 0.0002
AIC = -193.94  |  BIC = -154.15

Coeficientes significativos (p < 0.10):
  Intercepto                               beta=-0.8770  p=0.0481  **
  X1 — log PIB per capita                  beta=+0.1012  p=0.0084  ***
  X2 — Investimento / PIB                  beta=+1.7106  p=0.0905  *
  X5 — Crescimento Populacional            beta=-4.1055  p=0.0004  ***
